# Database update: add `source_text_name` to `Episodic` nodes

This notebook maps each `Episodic` node to one source text document by matching the episode `context` text to files listed in `data/sources.json`, then writes `source_text_name` into Neo4j.


In [1]:
import json
import re
from pathlib import Path
from difflib import SequenceMatcher

from neo4j import GraphDatabase

# Neo4j connection
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "knowledgegraph"
NEO4J_DATABASE = "neo4j"


In [2]:
# ---- Helpers ----
def normalize_text(text: str) -> str:
    text = text or ""
    text = text.replace("﻿", "")
    text = text.replace("–", "-").replace("—", "-").replace("−", "-")
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


def strip_section_header(text: str) -> str:
    # Graphiti chunks often start with: [Section: ...]
    text = text or ""
    return re.sub(r"^\s*\[section:[^\]]*\]\s*", "", text, flags=re.IGNORECASE).strip()


def snippet_hits(ctx_norm: str, doc_norm: str) -> int:
    if not ctx_norm:
        return 0
    n = len(ctx_norm)
    if n < 80:
        return int(ctx_norm in doc_norm)

    starts = [0, max(0, n // 3 - 40), max(0, 2 * n // 3 - 40)]
    hits = 0
    for s in starts:
        snippet = ctx_norm[s:s+80].strip()
        if snippet and snippet in doc_norm:
            hits += 1
    return hits


def token_overlap_score(ctx_norm: str, doc_norm: str) -> float:
    ctx_tokens = set(re.findall(r"[a-z0-9]+", ctx_norm))
    doc_tokens = set(re.findall(r"[a-z0-9]+", doc_norm))
    if not ctx_tokens:
        return 0.0
    return len(ctx_tokens & doc_tokens) / len(ctx_tokens)


def choose_source_for_context(ctx: str, docs_norm: dict) -> tuple[str | None, dict]:
    clean_ctx = strip_section_header(ctx)
    ctx_norm = normalize_text(clean_ctx)
    if not ctx_norm:
        return None, {"method": "empty_context"}

    # 1) strong exact/substr match
    exact_matches = [name for name, docn in docs_norm.items() if ctx_norm in docn]
    if len(exact_matches) == 1:
        return exact_matches[0], {"method": "exact_substring", "confidence": 1.0}
    if len(exact_matches) > 1:
        # Rare, but pick best by token overlap if ties happen
        best = None
        best_score = -1.0
        for name in exact_matches:
            score = token_overlap_score(ctx_norm, docs_norm[name])
            if score > best_score:
                best_score = score
                best = name
        return best, {"method": "exact_substring_tiebreak", "confidence": best_score}

    # 2) fallback scoring
    ranked = []
    sample_ctx = ctx_norm[:1200]
    for name, docn in docs_norm.items():
        hits = snippet_hits(ctx_norm, docn)
        overlap = token_overlap_score(sample_ctx, docn)
        # SequenceMatcher only on slices to keep runtime low
        seq = SequenceMatcher(None, sample_ctx, docn[:50000]).ratio()
        score = (hits * 0.5) + (overlap * 0.35) + (seq * 0.15)
        ranked.append((score, name, {"hits": hits, "overlap": overlap, "seq": seq}))

    ranked.sort(reverse=True, key=lambda x: x[0])
    best_score, best_name, details = ranked[0]

    # conservative threshold; below this we leave unmatched
    if best_score < 0.25:
        return None, {"method": "low_confidence", "best": ranked[:3]}

    return best_name, {"method": "scored_fallback", "score": best_score, "details": details, "top3": ranked[:3]}


In [3]:
# ---- Load source metadata + source text files ----
sources_path = Path("data/sources.json")
data_dir = sources_path.parent

with sources_path.open("r", encoding="utf-8") as f:
    source_meta = json.load(f)

entries = source_meta.get("entries", [])
assert entries, "No entries found in data/sources.json"

docs_raw = {}
docs_norm = {}
for e in entries:
    name = e["file_name"]
    txt_path = data_dir / f"{name}.txt"
    text = txt_path.read_text(encoding="utf-8")
    docs_raw[name] = text
    docs_norm[name] = normalize_text(text)

print("Loaded source docs:")
for k, v in docs_raw.items():
    print(f"- {k}: {len(v):,} chars")


Loaded source docs:
- haaland_2019: 6,268 chars
- transfermarkt_2019: 1,695 chars
- transfermarkt_2022: 1,482 chars


In [4]:
# ---- Fetch Episodic nodes ----
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

cypher_get_eps = """
MATCH (ep:Episodic)
RETURN
  elementId(ep) AS element_id,
  ep.uuid AS uuid,
  coalesce(ep.context, ep.content, ep.text, '') AS context
"""

with driver.session(database=NEO4J_DATABASE) as session:
    episodes = list(session.run(cypher_get_eps))

print(f"Fetched {len(episodes)} Episodic nodes")


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `context` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=6, column=15, offset=92>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 92, 'line': 6, 'column': 15}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (ep:Episodic)\nRETURN\n  elementId(ep) AS element_id,\n  ep.uuid AS uuid,\n  coalesce(ep.context, ep.content, ep.text, '') AS context\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `text` does not exist in 

Fetched 16 Episodic nodes


In [5]:
# ---- Match each episode to a source document ----
updates = []
diagnostics = []

for rec in episodes:
    element_id = rec["element_id"]
    uuid = rec["uuid"]
    ctx = rec["context"] or ""

    source_name, dbg = choose_source_for_context(ctx, docs_norm)

    diagnostics.append({
        "element_id": element_id,
        "uuid": uuid,
        "source_text_name": source_name,
        "method": dbg.get("method"),
        "debug": dbg,
    })

    if source_name is not None:
        updates.append({
            "element_id": element_id,
            "source_text_name": source_name,
        })

matched = len(updates)
total = len(episodes)
print(f"Matched: {matched}/{total} ({(matched/total*100) if total else 0:.1f}%)")

from collections import Counter
dist = Counter(u["source_text_name"] for u in updates)
print("Distribution:", dict(dist))

# Show a few unmatched samples for manual inspection
unmatched = [d for d in diagnostics if d["source_text_name"] is None]
print(f"Unmatched: {len(unmatched)}")
for row in unmatched[:10]:
    print("-", row["uuid"], row["method"])


Matched: 16/16 (100.0%)
Distribution: {'haaland_2019': 9, 'transfermarkt_2019': 4, 'transfermarkt_2022': 3}
Unmatched: 0


In [6]:
# ---- Write source_text_name to Episodic nodes ----
cypher_update = """
UNWIND $rows AS row
MATCH (ep)
WHERE elementId(ep) = row.element_id
SET ep.source_text_name = row.source_text_name
RETURN count(ep) AS updated_count
"""

with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run(cypher_update, rows=updates).single()

print(f"Updated Episodic nodes: {result['updated_count'] if result else 0}")
driver.close()


Updated Episodic nodes: 16


In [7]:
# Optional verification query
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
verify_q = """
MATCH (ep:Episodic)
RETURN ep.source_text_name AS source_text_name, count(*) AS n
ORDER BY n DESC
"""
with driver.session(database=NEO4J_DATABASE) as session:
    rows = list(session.run(verify_q))

for r in rows:
    print(r['source_text_name'], r['n'])

driver.close()


haaland_2019 9
transfermarkt_2019 4
transfermarkt_2022 3
